# Experiment A.03 — seed-999 smoke test

In [ ]:
import os
import subprocess
from pathlib import Path

R = Path.home() / "async-vla-latency-bench"
P = Path.home() / "LIBERO-plus"
N = Path.home() / "stage1-native"
MAIN = Path.home() / "experiment_a"
OUT = Path.home() / "experiment_a_smoke"
ID = Path.home() / "venv-stage1-id/bin/python"
OOD = Path.home() / "venv-stage1-ood/bin/python"

VAR = MAIN / "experiment_a_frozen_object_layout_variants.csv"
MAN = OUT / "experiment_a_smoke_manifest.csv"
AUDIT = OUT / "experiment_a_smoke_pairing_audit.csv"

GPU = "4"

OUT.mkdir(exist_ok=True)
gpu_state = subprocess.run(
    [
        "nvidia-smi",
        "-i",
        GPU,
        "--query-gpu=memory.used,utilization.gpu",
        "--format=csv,noheader,nounits",
    ],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

print("Physical GPU", GPU, "state:", gpu_state)

used, utilization = [
    int(value.strip()) for value in gpu_state.split(",")
]
if used >= 500 or utilization >= 5:
    raise SystemExit(
        f"STOP: physical GPU {GPU} is not idle: {gpu_state}"
    )

bench = "anonymous-source"






plus = subprocess.run(
    ["git", "-C", str(P), "rev-parse", "HEAD"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

subprocess.run(
    [
        str(ID),
        "-m",
        "async_vla_benchmark.scripts.make_experiment_a_smoke_manifest",
        "--variants",
        str(VAR),
        "--output",
        str(MAN),
        "--git-sha",
        bench,
        "--lerobot-git-sha",
        "2aba372b4e217cc47db28e0f836859b20d1456c9",
        "--libero-plus-git-sha",
        plus,
        "--model-revision",
        "8e174154ef5f6c60a8da12ae99c303d8963138c1",
    ],
    cwd=R,
    check=True,
)

base = os.environ.copy()
base.update(
    {
        "CUDA_VISIBLE_DEVICES": GPU,
        "MUJOCO_EGL_DEVICE_ID": GPU,
        "MUJOCO_GL": "egl",
        "PYOPENGL_PLATFORM": "egl",
        "MPLBACKEND": "Agg",
    }
)

subprocess.run(
    [
        str(ID),
        "-m",
        "async_vla_benchmark.scripts.resolve_stage3_initializations",
        "--config",
        str(R / "async_vla_benchmark/configs/experiment_a.yaml"),
        "--manifest",
        str(MAN),
        "--scene",
        "id",
        "--expected-rows",
        "8",
        "--expected-cells-per-key",
        "2",
        "--audit-output",
        str(AUDIT),
    ],
    cwd=R,
    env=base,
    check=True,
)

ood = base.copy()
ood.update(
    {
        "PYTHONPATH": str(P),
        "MAGICK_HOME": str(N),
        "PATH": str(N / "bin")
        + os.pathsep
        + ood.get("PATH", ""),
        "LD_LIBRARY_PATH": str(N / "lib")
        + os.pathsep
        + ood.get("LD_LIBRARY_PATH", ""),
    }
)

subprocess.run(
    [
        str(OOD),
        "-m",
        "async_vla_benchmark.scripts.resolve_stage3_initializations",
        "--config",
        str(R / "async_vla_benchmark/configs/experiment_a.yaml"),
        "--manifest",
        str(MAN),
        "--scene",
        "ood",
        "--expected-rows",
        "8",
        "--expected-cells-per-key",
        "2",
        "--audit-output",
        str(AUDIT),
    ],
    cwd=R,
    env=ood,
    check=True,
)

for scene, python, env in (
    ("id", ID, base),
    ("ood", OOD, ood),
):
    print(f"\nRunning Experiment A smoke shard: {scene}")

    command = [
        str(python),
        "-u",
        "-m",
        "async_vla_benchmark.scripts.run_experiment_a",
        "--config",
        str(R / "async_vla_benchmark/configs/experiment_a.yaml"),
        "--manifest",
        str(MAN),
        "--output-dir",
        str(OUT),
        "--scene",
        scene,
        "--resume",
        "--verbose",
    ]

    subprocess.run(
        command,
        cwd=R,
        env=env,
        check=True,
    )

subprocess.run(
    [
        str(ID),
        "-m",
        "async_vla_benchmark.scripts.validate_experiment_a_smoke",
        "--manifest",
        str(MAN),
        "--output-dir",
        str(OUT),
    ],
    cwd=R,
    env=base,
    check=True,
)

print(
    "STOP HERE: paste the smoke PASS line before notebook 04"
)